# 05 - Baseline B4: HistGradientBoostingClassifier

B4 ya está entrenado y versionado. Este cuaderno presenta los resultados y la configuración guardados en `results/baseline4/`; no ejecuta búsqueda ni reentrenamiento. La primera celda prepara este runtime de forma independiente. El procedimiento de entrenamiento permanece en `src/ptcg_wp/b4.py` y el registro experimental en el README de resultados.

In [ ]:
# Cada cuaderno prepara su propio runtime de Colab; no depende de 00_arranque ni de otra sesión.
from pathlib import Path
import sys, json, time, subprocess
from datetime import datetime, timezone
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo fetch origin 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo checkout 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo pull --ff-only origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / 'src') not in sys.path: sys.path.insert(0, str(REPO_DIR / 'src'))
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp.ingest import load_days
from ptcg_wp.features import feature_columns
from ptcg_wp.b4 import materialize_split, fit_b4_candidates_val
from ptcg_wp.pending_experiments import run_b4_ablation, run_h2, run_lightgbm
DATA_DIR = Path('/content/datos')
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(p.exists() for p in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(p.exists() for p in esperados), 'Faltan tablas procesadas; revisa Kaggle.'
schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62
SEED = 2026
SCENARIOS = ('A', 'B', 'C')
DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
RUN_INFO = {'commit': COMMIT, 'working_tree_dirty': DIRTY, 'seed': SEED, 'run_id_utc': RUN_ID}
RESULTS_DIR = REPO_DIR / 'results' / 'baseline4'
print('Tablas listas:', len(list(DATA_DIR.glob('estados_*.parquet'))), 'días | commit:', COMMIT,
      '| cambios locales:', DIRTY, '| corrida UTC:', RUN_ID)


## Métricas globales de validación

Se muestran por escenario; no se promedian. `test` permanece reservado para la etapa final.

In [ ]:
for escenario in ('A', 'B', 'C'):
    ruta = RESULTS_DIR / 'final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv'
    print(f'Escenario {escenario}')
    if ruta.exists():
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))
    else:
        print('No está disponible:', ruta)


## Configuración elegida

HistGradientBoostingClassifier, trial 20; `learning_rate=0.06943`, `max_leaf_nodes=31`, `min_samples_leaf=50` y `l2_regularization=4.2503`. Cada escenario se ajustó con su propio `train` y se evaluó en su `val`. Los modelos serializados están en `final_4000/escenario_X/`.

## Ablaciones por familia

Tres variantes omiten una familia: progreso, recursos/tablero o diferenciales. La configuración de B4 se mantiene fija; cada variante ajusta con el `train` propio y se mide en `val` de cada escenario. Son nueve ajustes finales, además de sus ajustes de vigilancia. `familias.json` registra las columnas exactas.

In [ ]:
from ptcg_wp.pending_experiments import run_b4_ablation
EJECUTAR_ABLACIONES = False
ESCENARIOS_ABLACIONES = ('A', 'B', 'C')
if EJECUTAR_ABLACIONES:
    candidate = pd.read_csv(REPO_DIR / 'results/baseline4/busqueda/b4_candidatas.csv')
    candidate = candidate.loc[candidate['trial'].eq(20)].copy()
    assert len(candidate) == 1
    for escenario in ESCENARIOS_ABLACIONES:
        out = DRIVE_ROOT / 'b4' / 'ablaciones' / RUN_ID / f'escenario_{escenario}'
        out.mkdir(parents=True, exist_ok=True)
        (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
        run_b4_ablation(DATA_DIR, escenario, FEATURES, candidate, out, seed=SEED)
        print('Guardado:', out)


## H2: estado completo frente a cuatro entradas

Reutiliza las métricas y el modelo B4 completo ya guardados. Ajusta la variante restringida con turno, primer jugador y los dos IDs activos. Los IDs se codifican como categorías con vocabulario de `train`; IDs no vistos en `val` se tratan como ausentes. Si hay más de 255 categorías, la celda se detiene para definir otra codificación. No se usa la variante de 64 entradas.

In [ ]:
from ptcg_wp.pending_experiments import run_h2
EJECUTAR_H2 = False
ESCENARIOS_H2 = ('A', 'B', 'C')
if EJECUTAR_H2:
    for escenario in ESCENARIOS_H2:
        out = DRIVE_ROOT / 'b4' / 'h2' / RUN_ID / f'escenario_{escenario}'
        out.mkdir(parents=True, exist_ok=True)
        (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
        run_h2(DATA_DIR, escenario, out, seed=SEED)
        completo = pd.read_csv(REPO_DIR / 'results/baseline4/final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv')
        restringido = pd.read_csv(out / 'h2_val.csv')
        print('B4 completo:', escenario); display(completo)
        print('H2 restringido:', escenario); display(restringido)
        print('Guardado:', out)


## Propuesta de comparación con LightGBM

Protocolo propuesto: una búsqueda TPE de 20 trials y 3 pliegues por partida, sobre partidas que son `train` en A, B y C; log loss como objetivo, vigilancia agrupada del 15 %, tope 2000 en búsqueda, 4000 en ajuste final y paciencia 30. Las finalistas por AUC, log loss y Brier se ajustan con el `train` propio y se miden por separado en `val`. Los parámetros tienen rangos análogos a B4, pero las implementaciones son distintas. El criterio para declarar una mejora clara sigue pendiente de acuerdo. Esta búsqueda es costosa.

In [ ]:
# Instalar SOLO si se va a ejecutar la comparación en este runtime.
# %pip install lightgbm==4.6.0


In [ ]:
from ptcg_wp.pending_experiments import run_lightgbm
EJECUTAR_LIGHTGBM = False
if EJECUTAR_LIGHTGBM:
    import lightgbm
    assert lightgbm.__version__ == '4.6.0', 'Instala la versión fijada en la celda anterior.'
    out = DRIVE_ROOT / 'lightgbm' / RUN_ID
    out.mkdir(parents=True, exist_ok=True)
    (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
    inicio = time.perf_counter()
    run_lightgbm(DATA_DIR, FEATURES, out, seed=SEED)
    (out / 'duracion.json').write_text(json.dumps({'seconds': time.perf_counter()-inicio}, indent=2))
    print('Guardado:', out)


## Resumen de la corrida actual

Muestra B4 completo y los resultados de ablaciones o LightGBM encontrados para el `RUN_ID` actual.

In [ ]:
# Después de ejecutar ablaciones o LightGBM, muestra sus métricas globales.
for escenario in ('A', 'B', 'C'):
    print('Escenario', escenario, '— B4 completo')
    full = pd.read_csv(REPO_DIR / 'results/baseline4/final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv')
    display(full.loc[full['etapa'].eq('global')].round(4))
    base = DRIVE_ROOT / 'b4' / 'ablaciones' / RUN_ID / f'escenario_{escenario}'
    for ruta in sorted(base.glob('sin_*/b4_val_finalists.csv')):
        print(ruta.parent.name)
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))
    ruta = DRIVE_ROOT / 'lightgbm' / RUN_ID / f'escenario_{escenario}' / 'lightgbm_val.csv'
    if ruta.exists():
        print('LightGBM — finalistas de CV')
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))
